# 표준 RAG 파이프라인 템플릿 (DOCX → Chroma, LangChain v1)

**표준 파이프라인**. 새 문서로 RAG를 빠르게 시작할 때 복사해서 사용

- 흐름: 문서로드 → 분할 → 임베딩 → 벡터스토어 → retriever → 답변생성
- 개념설명: [docs/langchain-packages.md](../../../../docs/langchain/langchain-packages.md) · 단계별 해설: [docs/langchain-rag-pipeline.md](../../../../docs/langchain/langchain-rag-pipeline.md)
- 바꿔 쓸 곳: `./tax.docx`(입력 문서), `persist_directory`, `collection_name`, `query`

# 1. 패키지 설치

In [ ]:
%pip install -U python-dotenv langchain langchain-openai langchain-chroma langchain-text-splitters langchain-community langchain-classic docx2txt chromadb

# 2. 문서 로드 + 분할

- `Docx2txtLoader` = DOCX 표준 로더 (아직 `langchain-community`에만 존재)
- 분할기는 `langchain_text_splitters`에서 import (`langchain.text_splitter`는 deprecated)

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

loader = Docx2txtLoader('./tax.docx')
document_list = loader.load_and_split(text_splitter=text_splitter)

# 3. 임베딩

- `OpenAIEmbeddings`는 `langchain_openai`에서 import (`langchain_community.embeddings`는 deprecated)

In [ ]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv()

embedding = OpenAIEmbeddings(model='text-embedding-3-large')  # 비용 절감 시 'text-embedding-3-small'

# 4. 벡터 스토어 (Chroma)

- `Chroma`는 `langchain_chroma`에서 import (`langchain_community.vectorstores`는 deprecated)

In [ ]:
from langchain_chroma import Chroma

# 최초 1회: 생성 + 영구 저장
database = Chroma.from_documents(
    documents=document_list,
    embedding=embedding,
    collection_name='chroma-tax',
    persist_directory='./chroma_db',
)

# 재사용 (2회차부터는 위를 주석 처리하고 아래 사용)
# database = Chroma(collection_name='chroma-tax', persist_directory='./chroma_db', embedding_function=embedding)

# 5. Retriever (유사도 검색)

In [ ]:
# similarity: 기본, 가장 유사한 k개 반환
retriever = database.as_retriever(search_type='similarity', search_kwargs={'k': 4})

# mmr: 다양성 + 유사도 균형 (실무 선호)
# retriever = database.as_retriever(search_type='mmr', search_kwargs={'k': 4, 'fetch_k': 20})

query = '연봉 5천만원인 거주자의 소득세는 얼마인가요?'
retriever.invoke(query)

# 6. 답변 생성 (RAG 체인)

- v1에서 `create_retrieval_chain`·`create_stuff_documents_chain`은 `langchain_classic.chains`에 위치 (deprecated 아님)
- 프롬프트는 `langchain.hub` 제거 → `langsmith.Client().pull_prompt`로 가져옴 (공개 프롬프트는 플래그 필요)

In [ ]:
from langsmith import Client
from langchain_openai import ChatOpenAI
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

prompt = Client().pull_prompt('langchain-ai/retrieval-qa-chat', dangerously_pull_public_prompt=True)

llm = ChatOpenAI(model='gpt-4o-mini')
combine_docs_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

In [ ]:
# 출력 dict의 키: input / context / answer
rag_chain.invoke({'input': query})